# HEATWIN — Phase 2 Physics Validation

Executable reference notebook. All plots are generated from the actual coupled energy balances. No PID or learned twin is used. For the full derivation, see `docs/mathematical_model.md`.

In [ ]:
from heatwin.config import load_config
from heatwin.models.plant import (PlantInputs, rhs, steady_state_for_target, steady_state_for_valve)
from heatwin.simulation.open_loop import InputStep, simulate_open_loop
from heatwin.visualization.charts import make_open_loop_figure
import numpy as np
import matplotlib.pyplot as plt

p = load_config()
eq = steady_state_for_target(p.setpoint, p)
print(eq)
print("Equilibrium derivative:", rhs(0, eq.state, PlantInputs.from_params(p, eq.u), p))

## A — hold computed nominal equilibrium for 300 s

In [ ]:
hold = simulate_open_loop(p, initial_command=eq.u, initial_state=eq.state)
print("Cold outlet max drift (°C):", max(abs(hold.tc_c - eq.tc)))
fig = make_open_loop_figure(hold, title="A — nominal equilibrium hold")
plt.show()

## B / C — increase or decrease hot-water flow at 60 s

In [ ]:
increase = simulate_open_loop(p, steps=[InputStep(60.0, u=0.50)])
decrease = simulate_open_loop(p, initial_command=0.50, steps=[InputStep(60.0, u=0.30)])
for title, frame in [("B — valve 40% to 50%", increase), ("C — valve 50% to 30%", decrease)]:
    print(title, "Tc at 300 s:", frame.tc_c.iloc[-1])
    fig = make_open_loop_figure(frame, title=title)
    plt.show()

## D — cold-water inlet disturbance at 100 s

In [ ]:
disturbance = simulate_open_loop(p, steps=[InputStep(100.0, tc_in=27.0)])
print("D final Tc:", disturbance.tc_c.iloc[-1])
fig = make_open_loop_figure(disturbance, title="D — cold inlet 30°C to 27°C")
plt.show()

## E — solver convergence (matched integer-second samples)

In [ ]:
coarse = simulate_open_loop(p, steps=[InputStep(60.0, u=0.5)], rtol=1e-7, atol=1e-9, sample_time_s=1.0)
fine = simulate_open_loop(p, steps=[InputStep(60.0, u=0.5)], rtol=1e-11, atol=1e-13, sample_time_s=0.25)
fine_common = fine[fine.time_s.isin(coarse.time_s)]
assert len(fine_common) == len(coarse)
for name in ["tc_c", "th_c", "xv"]:
    print(name, "max absolute difference:", np.max(np.abs(coarse[name].to_numpy() - fine_common[name].to_numpy())))